In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt

# 1. 读取密文（假设 cipher.txt 包含原始文本，包括空格和标点）
with open('cipher.txt', 'r', encoding='latin-1') as f:
    ciphertext_original = f.read()

# 为了方便分析，提取所有字母并转为小写
ciphertext_letters = re.sub(r'[^a-zA-Z]', '', ciphertext_original).lower()

# 定义重合指数函数
def index_of_coincidence(text):
    n = len(text)
    if n <= 1:
        return 0
    freq = Counter(text)
    ic = sum(f * (f - 1) for f in freq.values()) / (n * (n - 1))
    return ic

# 2. 寻找最佳密钥长度（尝试 1~20）
lengths = range(1, 21)
avg_ics = []
for l in lengths:
    cols = [''] * l
    for i, ch in enumerate(ciphertext_letters):
        cols[i % l] += ch
    avg_ic = sum(index_of_coincidence(col) for col in cols) / l
    avg_ics.append(avg_ic)

# 绘制图表，观察峰值
plt.plot(lengths, avg_ics, marker='o')
plt.axhline(y=0.065, color='r', linestyle='--', label='English IC (~0.065)')
plt.xlabel('Key Length')
plt.ylabel('Average Index of Coincidence')
plt.title('Finding Key Length for Vigenere Cipher')
plt.grid()
plt.show()

# 根据图表选择最可能的长度（例如峰值处）
# 假设你发现 key_len = 5 或 7 等，手动设定或自动选取最大峰值
key_len = lengths[avg_ics.index(max(avg_ics))]  # 简单取最高点
print(f'推测密钥长度: {key_len}')

# 3. 破解密钥
def break_vigenere_key(ciphertext, key_len):
    # 分成 key_len 个列
    cols = [''] * key_len
    for i, ch in enumerate(ciphertext):
        cols[i % key_len] += ch

    # 英语字母频率（常见顺序）
    english_freq = 'etaoinshrdlcumwfgypbvkjxqz'
    key = ''
    for col in cols:
        best_shift = 0
        best_score = -1
        for shift in range(26):
            # 假设该列用 shift 解密
            dec = ''.join(chr((ord(c) - ord('a') - shift) % 26 + ord('a')) for c in col)
            # 评分：统计解密后文本中常见字母出现的加权和
            score = sum(dec.count(letter) * (26 - i) for i, letter in enumerate(english_freq))
            if score > best_score:
                best_score = score
                best_shift = shift
        key += chr(ord('a') + best_shift)
    return key

key = break_vigenere_key(ciphertext_letters, key_len)
print(f'找到的密钥: {key}')

# 4. 解密原始文本（保留非字母字符）
def decrypt_vigenere(text, key):
    key = key.lower()
    result = []
    key_idx = 0
    for ch in text:
        if ch.isalpha():
            base = ord('a') if ch.islower() else ord('A')
            shift = ord(key[key_idx % len(key)]) - ord('a')
            if ch.isupper():
                result.append(chr((ord(ch) - ord('A') - shift) % 26 + ord('A')))
            else:
                result.append(chr((ord(ch) - ord('a') - shift) % 26 + ord('a')))
            key_idx += 1
        else:
            result.append(ch)
    return ''.join(result)

plaintext = decrypt_vigenere(ciphertext_original, key)
print('解密结果（前200字符）：')
print(plaintext[:200])

# 5. 保存到 cracked.txt
with open('cracked.txt', 'w', encoding='utf-8') as f:
    f.write(plaintext)

print('已保存 cracked.txt')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x93 in position 0: invalid start byte